In [ ]:
!pip install transformers datasets scikit-learn pandas torch

In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the dataset from Hugging Face
print("Loading 'chengxuphd/liar2' dataset...")
dataset = load_dataset("chengxuphd/liar2", split='train') # Use the train split
df = dataset.to_pandas()

# 2. Inspect the data
print("\n--- Real Dataset Head ---")
print(df.head())
print(f"\nTotal real examples: {len(df)}")
print("\nOriginal Label Distribution:")
print(df['label'].value_counts())

# 3. Simplify Labels (Critical Step)
# This dataset uses numbers from 0 to 5.
# FAKE (0): 0 (pants-fire), 1 (false)
# REAL (1): 2 (barely-true), 3 (half-true), 4 (mostly-true), 5 (true)
def simplify_label(label_int):
    if label_int in [0, 1]:
        return 0  # FAKE
    elif label_int in [2, 3, 4, 5]:
        return 1  # REAL
    else:
        return None # In case there are other labels

print("\nSimplifying labels...")
df['label'] = df['label'].map(simplify_label)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print("\n--- New Label Distribution ---")
print(df['label'].value_counts(normalize=True))

# 4. Rename 'statement' to 'text'
# This dataset might have a different column name, let's check
# Common names are 'statement', 'text', 'news'.
# If 'statement' fails, we'll try 'text'.
if 'statement' in df.columns:
    col_name = 'statement'
elif 'text' in df.columns:
    col_name = 'text'
else:
    print("ERROR: Could not find 'statement' or 'text' column.")
    # Stop here if we can't find the text
    raise ValueError("Dataset has no 'statement' or 'text' column")

print(f"Using column '{col_name}' as the input text.")
df_processed = df[[col_name, 'label']].rename(columns={col_name: 'text'})

# 5. Split into training and testing (stratified)
train_df, test_df = train_test_split(df_processed, test_size=0.2, random_state=42, stratify=df_processed['label'])

print(f"\nTraining examples: {len(train_df)}")
print(f"Testing examples: {len(test_df)}")

# 6. Convert back to Hugging Face Dataset objects
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

Loading 'chengxuphd/liar2' dataset...


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/19.0M [00:00<?, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/18369 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2297 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2296 [00:00<?, ? examples/s]


--- Real Dataset Head ---
      id  label                                          statement  \
0  13847      5  90 percent of Americans "support universal bac...   
1  13411      1  Last year was one of the deadliest years ever ...   
2  10882      0  Bernie Sanders's plan is "to raise your taxes ...   
3  20697      4  Voter ID is supported by an overwhelming major...   
4   6095      2  Says Barack Obama "robbed Medicare (of) $716 b...   

               date                                            subject  \
0   October 2, 2017  government regulation;polls and public opinion...   
1      May 19, 2017  after the fact;congress;criminal justice;histo...   
2  October 28, 2015                                              taxes   
3  December 8, 2021                                      voter id laws   
4   August 12, 2012         federal budget;history;medicare;retirement   

          speaker                                speaker_description  \
0     chris abele  Chris Abele is M

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
print(f"Loading tokenizer for: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

print("Tokenizing train dataset...")
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
print("Tokenizing test dataset...")
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True)

Loading tokenizer for: distilbert-base-uncased
Tokenizing train dataset...


Map:   0%|          | 0/14695 [00:00<?, ? examples/s]

Tokenizing test dataset...


Map:   0%|          | 0/3674 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Use 'binary' average for our 2-class problem
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # FAKE (0) vs REAL (1)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"\nModel loaded and moved to: {device}")

Loading model: distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model loaded and moved to: cuda


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,                   # 3 passes over the data
    learning_rate=2e-5,                   # Stable learning rate

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_dir="./logs",

    logging_strategy="steps",
    logging_steps=200,              # Log metrics every 200 steps
    eval_strategy="steps",
    eval_steps=200,                 # Run validation every 200 steps
    save_strategy="steps",
    save_steps=200,                 # Save a checkpoint every 200 steps

    load_best_model_at_end=True,
    metric_for_best_model="f1",     # We care most about F1-score
)

# Use the standard, reliable Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,  # Use the tokenized eval set
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

print("TrainingArguments set. Ready to train on 'chengxuphd/liar2' dataset.")

TrainingArguments set. Ready to train on 'chengxuphd/liar2' dataset.


/tmp/ipython-input-2514424875.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("--- Starting Model Training ---")

trainer.train()

print("--- Training Finished ---")

# After training, let's see the final evaluation
print("\n--- Final Model Evaluation ---")
eval_results = trainer.evaluate()
print(eval_results)

--- Starting Model Training ---


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
200,0.599100,0.555786,0.721557,0.765205,0.749213,0.781895
400,0.567200,0.530922,0.739521,0.797375,0.726746,0.883208
600,0.548800,0.533546,0.736255,0.799752,0.714813,0.907598
800,0.539700,0.531310,0.746053,0.793081,0.752209,0.838649
1000,0.523000,0.533936,0.743332,0.786989,0.759041,0.817073
1200,0.467800,0.532488,0.746053,0.807510,0.720810,0.917917
1400,0.475900,0.526249,0.748503,0.808537,0.724202,0.915103
1600,0.463500,0.542976,0.744692,0.790905,0.753611,0.832083
1800,0.451300,0.546020,0.749592,0.796999,0.752500,0.847092
2000,0.406900,0.570405,0.746326,0.799656,0.738095,0.872420


--- Training Finished ---

--- Final Model Evaluation ---


{'eval_loss': 0.5262488126754761, 'eval_accuracy': 0.7485029940119761, 'eval_f1': 0.8085370907583921, 'eval_precision': 0.7242019302152932, 'eval_recall': 0.9151031894934334, 'eval_runtime': 15.151, 'eval_samples_per_second': 242.492, 'eval_steps_per_second': 15.181, 'epoch': 3.0}


In [ ]:
from google.colab import files

MODEL_OUTPUT_DIR = "./my-final-model"

print(f"Saving final model and tokenizer to: {MODEL_OUTPUT_DIR}")
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("--- Model Saved Successfully ---")
!ls -l {MODEL_OUTPUT_DIR}

print(f"Zipping model files from {MODEL_OUTPUT_DIR}...")
!zip -r model.zip {MODEL_OUTPUT_DIR}

print("Zipping complete. Preparing for download...")
files.download('model.zip')

Saving final model and tokenizer to: ./my-final-model
--- Model Saved Successfully ---
total 262500
-rw-r--r-- 1 root root       563 Oct 27 18:36 config.json
-rw-r--r-- 1 root root 267832560 Oct 27 18:36 model.safetensors
-rw-r--r-- 1 root root       125 Oct 27 18:36 special_tokens_map.json
-rw-r--r-- 1 root root      1227 Oct 27 18:36 tokenizer_config.json
-rw-r--r-- 1 root root    711661 Oct 27 18:36 tokenizer.json
-rw-r--r-- 1 root root      5777 Oct 27 18:36 training_args.bin
-rw-r--r-- 1 root root    231508 Oct 27 18:36 vocab.txt
Zipping model files from ./my-final-model...
updating: my-final-model/ (stored 0%)
updating: my-final-model/vocab.txt (deflated 53%)
updating: my-final-model/config.json (deflated 45%)
updating: my-final-model/special_tokens_map.json (deflated 42%)
updating: my-final-model/tokenizer.json (deflated 71%)
updating: my-final-model/training_args.bin (deflated 53%)
updating: my-final-model/model.safetensors (deflated 8%)
updating: my-final-model/tokenizer_confi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>